# ML Modeler + ML Reviewer — Graph Wiring Review

Human-run notebook for the **graph wiring** slice that follows Step 5.

The prior Step 5 work delivered the modeler modes, the reviewer modes, and the `route_after_modeling_review` conditional router. This slice wires all of them into `orchestration/graph.py` so the modeling loop runs end-to-end after `ml_modeler_handoff`.

Run top to bottom to confirm:
- every new modeling-loop node is registered
- the full graph compiles
- every reviewer has a conditional edge driven by `route_after_modeling_review`
- every direct (non-reviewed) bridge has a plain edge
- the `ml_modeler_handoff` exit now points into the modeling loop (not to `END`)

In [ ]:
from pathlib import Path
import subprocess
from pprint import pprint

from multi_agent_ds.orchestration.graph import build_graph

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)

GRAPH = build_graph()
print("\nTotal nodes registered:", len(GRAPH.nodes))

def run_pytest(args):
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


## 1. Every modeling-loop node is registered

Expected: all 13 new nodes (5 reviewed modeler modes + 3 non-reviewed bridges + 5 reviewer verdict modes) appear alongside the 13 existing pre-modeling EDA nodes.

In [ ]:
modeling_loop_nodes = [
    "ml_modeler_baseline",
    "ml_reviewer_baseline_review",
    "ml_modeler_n_estimator_search",
    "ml_modeler_tune",
    "ml_reviewer_tuning_review",
    "ml_modeler_train_tuned",
    "ml_modeler_adjust_lr",
    "ml_reviewer_lr_adjustment_review",
    "ml_modeler_importance_review",
    "ml_modeler_feature_selection",
    "ml_reviewer_feature_selection_review",
    "ml_modeler_final_recommendation",
    "ml_reviewer_final_recommendation_review",
]

print(f"{'node':<44} registered")
print("-" * 60)
for node in modeling_loop_nodes:
    status = "OK" if node in GRAPH.nodes else "MISSING"
    print(f"{node:<44} {status}")

print("\nAll PRE-modeling + modeling node count:", len(GRAPH.nodes))


## 2. Full graph compiles end-to-end

The graph must compile without dangling edges, unresolved nodes, or cycles the runtime can't handle (LangGraph allows cycles, but they must terminate — the iteration cap is what guarantees termination here).

In [ ]:
compiled = GRAPH.compile()
print("Compiled type:", type(compiled).__name__)
print("Compiled graph nodes (subset):", list(compiled.nodes)[:10], "...")


## 3. Modeling loop edges (structural overview)

The loop's intended shape:

```
ml_modeler_handoff
  → ml_modeler_baseline → ml_reviewer_baseline_review
    ├── revise + under cap → ml_modeler_baseline (loop)
    └── advance → ml_modeler_n_estimator_search → ml_modeler_tune → ml_reviewer_tuning_review
                    ├── revise + under cap → ml_modeler_tune (loop)
                    └── advance → ml_modeler_train_tuned → ml_modeler_adjust_lr → ml_reviewer_lr_adjustment_review
                                    ├── revise + under cap → ml_modeler_adjust_lr (loop)
                                    └── advance → ml_modeler_importance_review → ml_modeler_feature_selection → ml_reviewer_feature_selection_review
                                                    ├── revise + under cap → ml_modeler_feature_selection (loop)
                                                    └── advance → ml_modeler_final_recommendation → ml_reviewer_final_recommendation_review
                                                                    ├── revise + under cap → ml_modeler_final_recommendation (loop)
                                                                    └── advance → END
```

Termination is guaranteed by `route_after_modeling_review` consulting `modeling_iteration < workflows.modeling.max_iterations`. Once any phase hits its cap, the router advances regardless of the revise verdict.

**Review question:** do the 3 direct bridges (`n_estimator_search → tune`, `train_tuned → adjust_lr`, `importance_review → feature_selection`) correctly skip review because those modes don't produce a decision the reviewer would verdict on?

## 4. Regression — router + graph + modeler + reviewer + earlier steps

Expected: **64 passed** (8 router incl. modeling-review + handoff-rewire; 4 graph; 17 modeler/reviewer; 14 cleaning/feature/prep + 21 data engineer adjacent from earlier steps — actual split may vary as new tests are added).

In [ ]:
run_pytest([
    "tests/test_langgraph_router.py",
    "tests/test_langgraph_graph.py",
    "tests/test_pre_modeling_review_agents.py",
    "tests/test_cleaning.py",
    "tests/test_feature_engineering.py",
    "tests/test_preparation_workflow.py",
])


## 5. Sign-off

If you are satisfied:
- no checklist item covers this explicitly (graph wiring was deferred from the main Step 5 checklist) — note the slice as complete in the plan progress log if you want
- then we commit the graph wiring and move to the final Step 5 follow-up: plumbing the reviewer's revision_questions into the modeler's next-turn prompt.

**Known follow-ups still open:**
- Modeler does not yet incorporate reviewer critique on loop-back (next slice).
- No end-to-end integration test runs the modeling loop with a real LLM (deferred — would need VCR/recorded fixtures).